# Lecture 6: Memory Management

> "Memory는 단순한 저장소가 아닙니다. **맥락을 유지하고, 학습하고, 개인화된 경험을 제공하기 위한 핵심 아키텍처**입니다."

AI Agent가 진정한 대화 능력을 갖추려면, 과거와 상호작용 하고 이를 기억하고 활용할 수 있어야 합니다.

## 환경 설정

- 실습 환경 구축을 위한 패키지 설치 (설치 후 세션을 다시 시작해야합니다.)

In [1]:
!pip install langgraph langchain-upstage langchain==1.1.3 rich

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.0
    Uninstalling langchain-1.2.0:
      Successfully uninstalled langchain-1.2.0
ERROR: pip's dependency resolver does not currently take into account all the packag

- Google Colab의 '보안 비밀번호(Secrets)' 메뉴에 등록한 API KEY를 환경 변수로 불러옵니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# 설정할 키 목록
keys = [
    'LANGSMITH_API_KEY', 'UPSTAGE_API_KEY'
]

for key in keys:
    value = os.getenv(key)
    if value is None:
        print(f"경고: {key}를 .env에서 찾을 수 없습니다.")
    else:
        os.environ[key] = value

# 고정값 설정
os.environ["LANGSMITH_TRACING_V2"] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = 'agentic-workflow'

- 실습에 필요한 라이브러리 로드 및 LLM 모델 초기화

In [3]:
from rich import print as rprint
from langchain.chat_models import init_chat_model

MODEL = "solar-pro2"
llm = init_chat_model(
    model=MODEL,
    temperature=0.0
)

## Checkpointer

#### 기억을 못하는 에이전트

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model = llm,
    system_prompt = "You are a helpful assistant"
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="안녕, 나는 파랑 청바지를 바지 중에 제일 좋아해")

response = agent.invoke(
    {"messages": [question]}
)
rprint(response['messages'][-1].content)

파랑 청바지를 좋아하시는군요! 🌙🖤  
청바지의 클래식한 매력이 정말 좋죠. 특히 **딥 블루**나 **와이드 핏** 청바지는 어떤 스타일에도 잘 어울리고, 계절을 
타지 않는 만능 아이템이에요.  

혹시 특정 브랜드나 스타일을 선호하시나요?  
예를 들어 **레트로한 빈티지 워싱**이나 **슬림 핏** 같은 걸 좋아하신다면, 추천 브랜드나 코디 팁도 알려드릴 수 
있어요! 😊  

> 💡 **TIP**: 파랑 청바지는 흰색 티셔츠나 베이지 카디건과 조합하면 무드 있는 캐주얼 룩이 완성돼요!  

관심 있는 스타일이 있다면 더 구체적으로 알려주세요!

In [6]:
question = HumanMessage(content="내가 제일 좋아하는 바지가 뭔지 알아?")

response = agent.invoke(
    {"messages": [question]}
)
rprint(response['messages'][-1].content)

당신이 가장 좋아하는 바지가 무엇인지 알려주시면, 그 바지에 대한 이야기를 함께 나눌 수 있을 것 같아요! 😊  

예를 들어,  
- **스타일** (청바지, 슬랙스, 조거 등)  
- **색상**이나 **디자인**  
- **특별한 추억**이나 **구매 계기**  
- **착용했을 때의 기분**  

등을 공유해 주시면 더 재미있게 대화할 수 있을 거예요.  
(혹시 비밀이라면 "비밀이에요~"라고 말해도 좋아요!)  

💡 *저는 청바지 중에서도 연한 워싱의 **미디움 웨이스트**를 좋아하는데, 편안하면서도 스타일링이 자유로워서 자주 
입어요!*  

당신의 최애 바지는 무엇인가요? 🤔👖

#### 기억을 하기 시작한 Agent

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = llm,
    system_prompt = "You are a helpful assistant",
    checkpointer = InMemorySaver()
)

In [8]:
question = HumanMessage(content="안녕, 나는 색깔 중 파란색을 제일 좋아해")
config = {"configurable": {"thread_id": "파란색 좋아하는 사람"}}

response = agent.invoke(
    {"messages": [question]} ,
    config
)
rprint(response['messages'][-1].content)

파란색을 좋아하시는군요! 🌊🌙  
파란색은 평화로움, 신뢰, 창의성을 상징하는 아름다운 색이에요.  

어떤 파란색 톤을 특히 좋아하시나요?  
- **하늘색**처럼 밝고 상쾌한 느낌?  
- **사파이어**처럼 깊고 고급스러운 느낌?  
- 아니면 **민트**처럼 산뜻한 파스텔 톤?  

색깔과 관련된 추억이나 좋아하는 파란색 아이템이 있다면 알려주세요! 함께 이야기 나누면 재미있을 것 같아요. 😊  

> 💙 *파란 하늘, 바다, 또는 특별한 파란색 물건을 보며 행복해지는 순간이 있다면 언제든 공유해주세요!*

In [9]:
question = HumanMessage(content="내가 제일 좋아하는 색깔이 뭔지 알아?")
config = {"configurable": {"thread_id": "파란색 좋아하는 사람"}}

response = agent.invoke(
    {"messages": [question]} ,
    config
)
rprint(response['messages'][-1].content)

네, 이미 알고 있어요! 😊  
**"파란색"**이 가장 좋아하는 색깔이라고 말씀해주셨잖아요.  

혹시 오늘 파란색을 보며 기분 전환하셨거나, 파란색과 관련된 재미있는 일이 있었나요?  
예를 들어, 파란 옷을 입었다거나, 하늘/바다 사진을 찍었다거나, 파란색 아이템을 구매하셨다거나…  

색깔은 기분과 감정에 큰 영향을 주니까, 좋아하는 파란색을 활용한 작은 즐거움을 나누는 것도 좋을 것 같아요! 🌉🎨  

> 💙 *파란색의 어떤 점이 특히 마음에 드는지 궁금해요!*

## 도구를 사용한 기억 관리

Context Window는 모든 기억을 저장하기에 너무 짧기도 하고,<br>
Context Window가 너무 길어지면, 맥락 안에 있어도 단기 기억 상실증이 올 수 있습니다.<br>
State에 저장해두고, 필요할 때마다 정보를 꺼내서 가져오면 어떨까요?

- **Context Rot**
    - 전체 토큰 수가 기술적 한도(예: 100만 토큰)에 충분함에도 불구하고 컨텍스트 창이 채워질수록 LLM의 성능이 저하되는 현상

### Update Memory

In [10]:
from langchain.agents import AgentState

class PersonalState(AgentState):
    favourite_color: str

In [11]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_color(favourite_color: str, runtime: ToolRuntime) -> Command:
    """Update the favourite color of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_color": favourite_color,
        "messages": [ToolMessage("Successfully updated favourite color", tool_call_id=runtime.tool_call_id)]}
        )


In [12]:
agent = create_agent(
    model = llm,
    system_prompt = "You are a helpful coordinator who assists users with their fashion",
    tools = [update_favourite_color],
    checkpointer = InMemorySaver(),
    state_schema = PersonalState
)

In [13]:
question = HumanMessage(content="안녕, 나는 색깔 중 파란색을 제일 좋아해")
config = {"configurable": {"thread_id": "파란색을 좋아하던 사람"}}

response = agent.invoke(
    {"messages": [question]} ,
    config
)

In [14]:
rprint(response)
rprint(response['messages'][-1].content)

{
    'messages': [
        HumanMessage(
            content='안녕, 나는 색깔 중 파란색을 제일 좋아해',
            additional_kwargs={},
            response_metadata={},
            id='66c16826-fd4c-4f0c-b4de-1246ff4e3cab'
        ),
        AIMessage(
            content='[사용자가 직접 "파란색"을 가장 좋아하는 색으로 명시했기 때문에, 이 정보를 시스템에 
업데이트하는 것이 필수적입니다. 다른 함수 호출은 필요하지 않습니다.]',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 48,
                    'prompt_tokens': 527,
                    'total_tokens': 575,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'solar-pro2-251215',
                'system_fingerprint': None,
                'id': 'eb1baf42-821e-47c7-9e16-56bc43bcd9f6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019b91ee-8761-7632-b41f-a92755d2f213-0',
            tool_calls=[
                {
                    'name': 'update_favourite_color',
                    'args': {'favourite_color': '파란색'},
                    'id': 'chatcmpl-tool-3df6978ed6d24b279aaf1457bec6a4b3',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 527,
                'output_tokens': 48,
                'total_tokens': 575,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Successfully updated favourite color',
            name='update_favourite_color',
            id='e16eb6b6-4d59-48fa-b14c-7a32328f6f8f',
            tool_call_id='chatcmpl-tool-3df6978ed6d24b279aaf1457bec6a4b3'
        ),
        AIMessage(
            content='알겠습니다! 사용자의 선호 색상이 "파란색"으로 업데이트되었습니다.  \n앞으로 패션 조언이나 코디
추천 시 이 정보를 반영하겠습니다. 😊  \n\n예를 들어, "파란색 계열의 스타일링"이나 "파란색 액세서리 활용법" 등 
구체적인 요청이 있으면 언제든지 알려주세요!',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 56,
                    'prompt_tokens': 592,
                    'total_tokens': 648,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'solar-pro2-251215',
                'system_fingerprint': None,
                'id': 'fbd198b0-f65a-4813-b377-8cd8c595c7ee',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019b91ee-8d7a-7b41-9283-3e85677aed1a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 592,
                'output_tokens': 56,
                'total_tokens': 648,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'favourite_color': '파란색'
}

알겠습니다! 사용자의 선호 색상이 "파란색"으로 업데이트되었습니다.  
앞으로 패션 조언이나 코디 추천 시 이 정보를 반영하겠습니다. 😊  

예를 들어, "파란색 계열의 스타일링"이나 "파란색 액세서리 활용법" 등 구체적인 요청이 있으면 언제든지 알려주세요!

In [15]:
question = HumanMessage(content="나한테 어떤 옷이 어울릴 것 같아?")
config = {"configurable": {"thread_id": "파란색을 좋아하던 사람"}}

response = agent.invoke(
    {
        "messages": [question],
        "favourite_color": "yellow"
    },
    config
)
rprint(response)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='안녕, 나는 색깔 중 파란색을 제일 좋아해',
            additional_kwargs={},
            response_metadata={},
            id='66c16826-fd4c-4f0c-b4de-1246ff4e3cab'
        ),
        AIMessage(
            content='[사용자가 직접 "파란색"을 가장 좋아하는 색으로 명시했기 때문에, 이 정보를 시스템에 
업데이트하는 것이 필수적입니다. 다른 함수 호출은 필요하지 않습니다.]',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 48,
                    'prompt_tokens': 527,
                    'total_tokens': 575,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'solar-pro2-251215',
                'system_fingerprint': None,
                'id': 'eb1baf42-821e-47c7-9e16-56bc43bcd9f6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019b91ee-8761-7632-b41f-a92755d2f213-0',
            tool_calls=[
                {
                    'name': 'update_favourite_color',
                    'args': {'favourite_color': '파란색'},
                    'id': 'chatcmpl-tool-3df6978ed6d24b279aaf1457bec6a4b3',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 527,
                'output_tokens': 48,
                'total_tokens': 575,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Successfully updated favourite color',
            name='update_favourite_color',
            id='e16eb6b6-4d59-48fa-b14c-7a32328f6f8f',
            tool_call_id='chatcmpl-tool-3df6978ed6d24b279aaf1457bec6a4b3'
        ),
        AIMessage(
            content='알겠습니다! 사용자의 선호 색상이 "파란색"으로 업데이트되었습니다.  \n앞으로 패션 조언이나 코디
추천 시 이 정보를 반영하겠습니다. 😊  \n\n예를 들어, "파란색 계열의 스타일링"이나 "파란색 액세서리 활용법" 등 
구체적인 요청이 있으면 언제든지 알려주세요!',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 56,
                    'prompt_tokens': 592,
                    'total_tokens': 648,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'solar-pro2-251215',
                'system_fingerprint': None,
                'id': 'fbd198b0-f65a-4813-b377-8cd8c595c7ee',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019b91ee-8d7a-7b41-9283-3e85677aed1a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 592,
                'output_tokens': 56,
                'total_tokens': 648,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        HumanMessage(
            content='나한테 어떤 옷이 어울릴 것 같아?',
            additional_kwargs={},
            response_metadata={},
            id='0f462719-0268-454e-92dd-0237351b8648'
        ),
        AIMessage(
    

{
    'messages': [
        HumanMessage(
            content='안녕, 나는 색깔 중 파란색을 제일 좋아해',
            additional_kwargs={},
            response_metadata={},
            id='66c16826-fd4c-4f0c-b4de-1246ff4e3cab'
        ),
        AIMessage(
            content='[사용자가 직접 "파란색"을 가장 좋아하는 색으로 명시했기 때문에, 이 정보를 시스템에 
업데이트하는 것이 필수적입니다. 다른 함수 호출은 필요하지 않습니다.]',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 48,
                    'prompt_tokens': 527,
                    'total_tokens': 575,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'solar-pro2-251215',
                'system_fingerprint': None,
                'id': 'eb1baf42-821e-47c7-9e16-56bc43bcd9f6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019b91ee-8761-7632-b41f-a92755d2f213-0',
            tool_calls=[
                {
                    'name': 'update_favourite_color',
                    'args': {'favourite_color': '파란색'},
                    'id': 'chatcmpl-tool-3df6978ed6d24b279aaf1457bec6a4b3',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 527,
                'output_tokens': 48,
                'total_tokens': 575,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Successfully updated favourite color',
            name='update_favourite_color',
            id='e16eb6b6-4d59-48fa-b14c-7a32328f6f8f',
            tool_call_id='chatcmpl-tool-3df6978ed6d24b279aaf1457bec6a4b3'
        ),
        AIMessage(
            content='알겠습니다! 사용자의 선호 색상이 "파란색"으로 업데이트되었습니다.  \n앞으로 패션 조언이나 코디
추천 시 이 정보를 반영하겠습니다. 😊  \n\n예를 들어, "파란색 계열의 스타일링"이나 "파란색 액세서리 활용법" 등 
구체적인 요청이 있으면 언제든지 알려주세요!',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 56,
                    'prompt_tokens': 592,
                    'total_tokens': 648,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'solar-pro2-251215',
                'system_fingerprint': None,
                'id': 'fbd198b0-f65a-4813-b377-8cd8c595c7ee',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019b91ee-8d7a-7b41-9283-3e85677aed1a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 592,
                'output_tokens': 56,
                'total_tokens': 648,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        HumanMessage(
            content='나한테 어떤 옷이 어울릴 것 같아?',
            additional_kwargs={},
            response_metadata={},
            id='0f462719-0268-454e-92dd-0237351b8648'
        ),
        AIMessage(
    

In [16]:
question = HumanMessage(content="나는 무슨 색을 제일 좋아해?")
config = {"configurable": {"thread_id": "파란색을 좋아하던 사람"}}

response = agent.invoke(
    {
        "messages": [question],
    },
    config
)

rprint(response['messages'][-1].content)

사용자는 이전에 "파란색"을 가장 좋아하는 색으로 직접 언급하셨고, 해당 정보는 시스템에 업데이트된 상태입니다.  

**답변**:  
"파란색"을 가장 좋아하신다고 알려주셨습니다! 😊  

추가로 다른 색상 선호도나 패션 관련 질문이 있으면 언제든지 말씀해 주세요.

'favourite_color'는 'yellow'가 됐지만 LLM은 이를 인지하지 못하는 걸로 보입니다.

### Get Memory

Agent가 Agent 모르게 바뀐 정보를 인지할 수 있는지 확인해봅니다.

In [20]:
@tool
def read_favourite_color(runtime: ToolRuntime) -> str:
    """Read the favourite color of the user from the state."""
    try:
        return runtime.state["favourite_color"]
    except KeyError:
        return "No favourite color found in state"

agent = create_agent(
    llm,
    tools=[update_favourite_color, read_favourite_color],
    checkpointer=InMemorySaver(),
    state_schema=PersonalState
)

In [21]:
question = HumanMessage(content="안녕, 나는 색깔 중 파란색을 제일 좋아해")
config = {"configurable": {"thread_id": "노란색을 좋아할 사람"}}

response = agent.invoke(
    {"messages": [question]} ,
    config
)

In [22]:
question = HumanMessage(content="나한테 어떤 옷이 어울릴 것 같아?")
config = {"configurable": {"thread_id": "노란색을 좋아할 사람"}}

response = agent.invoke(
    {
        "messages": [question],
        "favourite_color": "yellow"
    },
    config
)

In [ ]:
# rprint(response)
# # rprint(response['messages'][-1].content)

In [23]:
question = HumanMessage(content="나는 무슨 색을 제일 좋아해?")
config = {"configurable": {"thread_id": "노란색을 좋아할 사람"}}

response = agent.invoke(
    {
        "messages": [question],
    },
    config
)

rprint(response['messages'][-1].content)

최애 색상이 **"yellow"**로 성공적으로 업데이트되었습니다! 🟡  

이제 노란색 계열 옷이나 액세서리 추천을 원하시거나, 다른 색상 관련 질문이 있으면 언제든지 말씀해 주세요! 😊

In [ ]:
# question = HumanMessage(content="아니야 너 까먹은거 같아 내 정보를 확인해봐")
# config = {"configurable": {"thread_id": "노란색을 좋아할 사람"}}

# response = agent.invoke(
#     {
#         "messages": [question],
#     },
#     config
# )

# rprint(response['messages'][-1].content)

## Message Summarizing

대화가 길어지면 컨텍스트 윈도우를 초과할 수 있습니다.<br>
메세지를 요약하여 관리해봅니다.

In [24]:
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="solar-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [25]:
from langchain.messages import HumanMessage, AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="요즘 여자친구랑 자주 싸우는데 어떻게 해야 할지 모르겠어."),
        AIMessage(content="주로 어떤 문제로 다투시나요? 구체적인 상황을 말씀해 주시면 조언해 드릴 수 있어요."),
        HumanMessage(content="연락 문제 때문이야. 나는 일이 바빠서 답장이 늦을 때가 많은데, 여자친구는 그걸 서운해해."),
        AIMessage(content="연락 빈도 차이로 서운함이 쌓였나 보네요. 상대방은 연락을 '나에 대한 관심'으로 느낄 수 있거든요."),
        HumanMessage(content="그렇구나. 내가 업무 중이라 정말 바쁠 때는 어떻게 대처하는 게 현명할까?"),
        AIMessage(content="상황을 미리 공유하는 게 좋아요. '지금 회의 들어가서 2시간 뒤에 연락할게'처럼 예측 가능하게 해주는 거죠."),
        HumanMessage(content="그런데 미리 말해도 회의가 길어져서 약속한 시간을 못 지키면 상황이 더 나빠지지 않을까?"),
        AIMessage(content="그럴 때는 솔직하게 상황을 설명하고, 기다려준 것에 대해 고마움을 표현하는 진정성이 중요해요."),
        HumanMessage(content="만약 네가 내 입장이라면, 오늘 밤에 여자친구한테 어떤 식으로 먼저 말을 꺼낼 것 같아?"),
    ]},
    {"configurable": {"thread_id": "1"}}
)

# rprint(response)

In [ ]:
rprint(response["messages"][0].content)

Here is a summary of the conversation to date:

연락 빈도 차이로 서운함이 쌓였나 보네요. 상대방은 연락을 '나에 대한 관심'으로 느낄 수 있거든요. 상황을 미리 
공유하는 게 좋아요. '지금 회의 들어가서 2시간 뒤에 연락할게'처럼 예측 가능하게 해주는 거죠. 그럴 때는 솔직하게 
상황을 설명하고, 기다려준 것에 대해 고마움을 표현하는 진정성이 중요해요.

## Message Trimming

필요 없는 대화는 잘라냅니다.

In [26]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    to_remove = []
    for m in messages:
        # 1. ToolMessage(도구 결과)는 무조건 삭제 대상
        if isinstance(m, ToolMessage):
            to_remove.append(m.id)

        # 2. tool_calls가 포함된 AIMessage(도구 요청)도 삭제 대상
        # (단, 일반 답변과 섞여있을 수 있으므로 content가 없는 경우 위주로 삭제)
        elif isinstance(m, AIMessage) and m.tool_calls:
            to_remove.append(m.id)

    if not to_remove:
        return None

    # id가 존재하는 메시지만 RemoveMessage로 반환
    return {"messages": [RemoveMessage(id=m_id) for m_id in to_remove if m_id]}

In [27]:
agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [28]:
from langchain.messages import HumanMessage, AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="오늘 삼성전자 주가는 어때?"),
        AIMessage(content="", tool_calls=[{"name": "get_stock_price", "args": {"ticker": "005930"}, "id": "t1"}]),
        ToolMessage(content='{"price": 75400, "change": "+1.2%"}', tool_call_id="t1", id="m1"),
        AIMessage(content="오늘 삼성전자는 전일 대비 1.2% 상승한 75,400원입니다. 최근 반도체 업황 개선 기대감이 반영된 것으로 보입니다."),
        HumanMessage(content="관련된 최신 뉴스도 좀 찾아봐줄래?"),
        AIMessage(content="", tool_calls=[{"name": "search_news", "args": {"query": "삼성전자 반도체"}, "id": "t2"}]),
        ToolMessage(content='{"headline": "삼성전자, 차세대 HBM 양산 가속화...", "sentiment": "positive"}', tool_call_id="t2", id="m2"),
        AIMessage(content="최신 뉴스에 따르면, 삼성전자가 차세대 HBM(고대역폭메모리) 양산을 가속화하고 있다는 긍정적인 소식이 있습니다."),
        HumanMessage(content="그럼 지금 매수하는 게 좋을까? 내 자산 포트폴리오랑 비교해서 알려줘."),
    ]},
    {"configurable": {"thread_id": "1"}}
)

rprint(response)

{
    'messages': [
        HumanMessage(
            content='오늘 삼성전자 주가는 어때?',
            additional_kwargs={},
            response_metadata={},
            id='9c077c01-707f-49f6-ace0-18582cef490b'
        ),
        AIMessage(
            content='오늘 삼성전자는 전일 대비 1.2% 상승한 75,400원입니다. 최근 반도체 업황 개선 기대감이 반영된 
것으로 보입니다.',
            additional_kwargs={},
            response_metadata={},
            id='d4b0e537-1355-40eb-b530-2b2acd3f466b',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='관련된 최신 뉴스도 좀 찾아봐줄래?',
            additional_kwargs={},
            response_metadata={},
            id='a4f31971-c246-4dad-811a-6139e3a64b64'
        ),
        AIMessage(
            content='최신 뉴스에 따르면, 삼성전자가 차세대 HBM(고대역폭메모리) 양산을 가속화하고 있다는 긍정적인 
소식이 있습니다.',
            additional_kwargs={},
            response_metadata={},
            id='fbef0b11-eca6-4622-a85f-b092d1da4dc3',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='그럼 지금 매수하는 게 좋을까? 내 자산 포트폴리오랑 비교해서 알려줘.',
            additional_kwargs={},
            response_metadata={},
            id='d5f271e9-1ae5-4528-8ac8-387deac3d734'
        ),
        AIMessage(
            content='투자 결정을 내리기 전에 몇 가지 중요한 요소를 고려해야 합니다. 현재 삼성전자의 주가 동향과 
시장 상황을 종합적으로 분석해 드리겠습니다.\n\n### 1. **삼성전자 최근 동향 (2024년 7월 기준)**\n   - **주가**: 전일
대비 1.2% 상승한 75,400원 (참고: 실시간 데이터 아님).\n   - **HBM3E 양산**: AI 반도체 수요 증가로 HBM(고대역폭 
메모리) 시장 점유율 확대 중. 2024년 2분기 HBM 매출이 전년 대비 10배 증가했다는 보도 있음.\n   - **반도체 업황 
회복**: D램·낸드플래시 가격 상승세가 지속되며, 2024년 하반기 실적 개선 전망.\n   - **리스크**: 글로벌 경기 
불확실성, 미중 기술 갈등, 경쟁사(TSMC, SK하이닉스)와의 경쟁.\n\n### 2. **포트폴리오 비교 분석 (가정)**\n   - **현재
포트폴리오**가 **방어적 자산(채권, 금 등) 60% + 성장주(테크) 30% + 현금 10%**로 구성되었다면:\n     - 삼성전자는 
**테크 섹터 내 반도체**에 집중되어 있으므로, 기존 포트폴리오의 **성장주 비중**을 재조정해야 할 수 있음.\n     - 
반도체 업종이 현재 **사이클적 회복기**에 있지만, 단기 변동성 가능성 존재.\n\n### 3. **매수 고려 시 체크포인트**\n  
- **투자 목적**: 단기 트레이딩 vs. 장기 투자.\n     - *단기*: 기술적 분석(지지/저항선, 거래량) 필요. 현재 75,000원 
근처에서 **20일 이동평균선**을 지지하는지 확인.\n     - *장기*: 반도체 산업 성장성(인공지능, 데이터센터 수요)을 
고려할 때 매력적일 수 있음.\n   - **포트폴리오 균형**: 테크 섹터 비중이 이미 높다면, **분산 투자**를 위해 다른 
산업(헬스케어, 재생에너지) 추가 고려.\n   - **리스크 관리**: 단일 종목에 5% 이상 투자하지 않는 것이 일반적. 현재 
포트폴리오에서 삼성 비중이 얼마인지 확인 필요.\n\n### 4. **추천 전략**\n   - **분할 매수**: 현재 가격대에서 30%, 
70,000원 돌파 시 30%, 75,000원 재테스트 시 40% 등 단계적 진입.\n   - **손절 기준**: 70,000원 이하로 떨어질 경우 
재평가.\n   - **대안**: 반도체 ETF(예: SOXX)로 분산 투자하거나, SK하이닉스와 비교해 밸류에이션 검토.\n\n### 5. 
**참고 자료**\n   - **증권사 리포트**: 한국투자증권 "삼성전자, HBM 수혜로 2분기 실적 기대치 상회 전망" 
(2024.07.10).\n   - **시장 지표**: 미국 필라델피아 반도체 지수(SOX) 최근 5% 상승, 글로벌 반도체 수요 증가 
신호.\n\n> 💡 **결론**: 단기 변동성은 있으나, 중장기적으로 반도체 업황 회복과 HBM 수요 증가를 고려할 때 **점진적 
매수**는 유효할 수 있습니다. 다만, 포트폴리오의 기존 테크 비중을 고려해 **5% 내외**로 제한하는 것이 안전합니다. 
실시간 데이터와 개인 투자 성향을 반드시 확인하세요!  \n\n(※ 주의: 이 분석은 참고용이며, 투자 책임은 본인에게 
있습니다. 정확한 데이터는 금융 앱/증권사 리포트에서 재확인하세요.)',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 692,
                    'prompt_tokens': 130,
                    'total_tokens': 822,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'solar-pro2-251215',
                'system_fingerprint': None,
                'id': '130c6793-1b89-4e6f-909d-eef4424eff60',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019b9204-a489-7191-9f3f-a2a93dfab6f5-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens':